In [12]:
!pip install transformers datasets accelerate scikit-learn -q

import json
import os
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

# --- IMPORTANT: Change this to match the folder name you chose when uploading ---
DATA_FILE = "/kaggle/input/datasets/raghavkapil/router-training-data2/labeled_dataset.json"
OUTPUT_DIR = "/distilbert-complexity"

BASE_MODEL = "distilbert-base-uncased"
LABEL2ID = {"weak": 0, "strong": 1}
ID2LABEL = {0: "weak", 1: "strong"}

# 1. Load Data
with open(DATA_FILE, "r", encoding="utf-8") as f:
    items = json.load(f)

texts = [item["prompt"] for item in items]
labels = [LABEL2ID[item["label"]] for item in items]

from datasets import ClassLabel

ds = Dataset.from_dict({"text": texts, "label": labels})
# FIX: Explicitly tell Hugging Face this is a categorical label column
ds = ds.cast_column("label", ClassLabel(num_classes=2, names=["weak", "strong"]))
# Now it will split perfectly
splits = ds.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
dataset = DatasetDict({"train": splits["train"], "validation": splits["test"]})

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
def tokenize(examples):
    return tokenizer(examples["text"], padding=False, truncation=True, max_length=512)
tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

# 3. Model
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)



Casting the dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
# 4. Train
training_args = TrainingArguments(
    output_dir="./checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    warmup_ratio=0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    fp16=False, 
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting training on GPU...")
trainer.train()

# 5. Save & Evaluate
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\n--- Final Evaluation ---")
preds_output = trainer.predict(tokenized["validation"])
preds = np.argmax(preds_output.predictions, axis=-1)
print(classification_report(preds_output.label_ids, preds, target_names=["weak", "strong"]))

# Zip the output so you can download it easily
!zip -r model.zip ./distilbert-complexity
print("Done! You can now download model.zip from the right sidebar.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training on GPU...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.379199,0.541667,0.351351
2,No log,1.304875,0.700000,0.648323
3,No log,1.227043,0.941667,0.940741
4,No log,1.145476,0.983333,0.983217
5,No log,1.067829,0.983333,0.983217
6,No log,1.000895,0.975000,0.974859
7,No log,0.949847,0.975000,0.974859
8,No log,0.913397,0.975000,0.974859
9,No log,0.892888,0.975000,0.974859
10,No log,0.885826,0.975000,0.974859


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Evaluation ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

        weak       0.98      0.98      0.98        65
      strong       0.98      0.98      0.98        55

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120

updating: distilbert-complexity/ (stored 0%)
updating: distilbert-complexity/training_args.bin (deflated 53%)
updating: distilbert-complexity/config.json (deflated 50%)
updating: distilbert-complexity/model.safetensors (deflated 8%)
updating: distilbert-complexity/tokenizer_config.json (deflated 42%)
updating: distilbert-complexity/tokenizer.json (deflated 71%)
Done! You can now download model.zip from the right sidebar.


In [17]:
from IPython.display import FileLink

# Put the name of the file you want to download inside the quotes
FileLink(r'model.zip')

/kaggle/working/model.zip